# Cheque Image Parser

This notebook reads cheque images from the volume `ramin_serverless_aws_catalog.symcor.cheques`, parses them using `ai_parse_document()`, extracts key fields, and saves results to a table.

In [0]:
# Read all cheque images from the volume and parse them with ai_parse_document
parsed_cheques_df = spark.sql("""
  SELECT
    _metadata.file_path,
    _metadata.file_name,
    _metadata.file_size,
    _metadata.file_modification_time,
    ai_parse_document(content, map('version', '2.0')) AS parsed_content
  FROM read_files(
    '/Volumes/ramin_serverless_aws_catalog/symcor/cheques/',
    format => 'binaryFile'
  )
""")

# Create temporary view for downstream processing
parsed_cheques_df.createOrReplaceTempView("parsed_cheques")

# Display sample results
display(parsed_cheques_df.limit(5))

In [0]:
# Extract and concatenate all text elements from the parsed documents
cheque_text_df = spark.sql("""
  SELECT
    file_name,
    file_path,
    concat_ws('\n',
      transform(
        try_cast(parsed_content:document:elements AS ARRAY<VARIANT>),
        element -> try_cast(element:content AS STRING)
      )
    ) AS full_text,
    parsed_content
  FROM parsed_cheques
  WHERE try_cast(parsed_content:error_status AS STRING) IS NULL
""")

# Create temporary view
cheque_text_df.createOrReplaceTempView("cheque_text")

# Display extracted text
display(cheque_text_df.select("file_name", "full_text").limit(3))

In [0]:
# Use ai_extract to identify and extract key cheque fields from the text
cheque_fields_df = spark.sql("""
  SELECT
    file_name,
    file_path,
    ai_extract(
      full_text,
      array(
        'payee_name',
        'amount',
        'date',
        'cheque_number',
        'account_number',
        'routing_number',
        'memo'
      )
    ) AS extracted_fields
  FROM cheque_text
""")

# Create temporary view
cheque_fields_df.createOrReplaceTempView("cheque_fields")

# Display extracted fields
display(cheque_fields_df.limit(5))

In [0]:
# Create the final table with all extracted cheque information
final_table_df = spark.sql("""
  CREATE OR REPLACE TABLE ramin_serverless_aws_catalog.symcor.parsed_cheques AS
  SELECT
    file_name,
    file_path,
    extracted_fields.payee_name AS payee_name,
    extracted_fields.amount AS amount,
    extracted_fields.date AS cheque_date,
    extracted_fields.cheque_number AS cheque_number,
    extracted_fields.account_number AS account_number,
    extracted_fields.routing_number AS routing_number,
    extracted_fields.memo AS memo,
    current_timestamp() AS processed_at
  FROM cheque_fields
""")

print("✓ Table created successfully: ramin_serverless_aws_catalog.symcor.parsed_cheques")

# Display the results
results_df = spark.table("ramin_serverless_aws_catalog.symcor.parsed_cheques")
display(results_df)

In [0]:
# Generate summary statistics for the parsed cheques
summary_df = spark.sql("""
  SELECT 
    COUNT(*) AS total_cheques_processed,
    COUNT(DISTINCT payee_name) AS unique_payees,
    COUNT(CASE WHEN amount IS NOT NULL THEN 1 END) AS cheques_with_amount,
    COUNT(CASE WHEN cheque_date IS NOT NULL THEN 1 END) AS cheques_with_date,
    COUNT(CASE WHEN cheque_number IS NOT NULL THEN 1 END) AS cheques_with_number,
    COUNT(CASE WHEN account_number IS NOT NULL THEN 1 END) AS cheques_with_account
  FROM ramin_serverless_aws_catalog.symcor.parsed_cheques
""")

print("📊 Cheque Parsing Summary:")
display(summary_df)